In [1]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def f(x):
    return 3*x**2 - 4*x +5

In [3]:
f(3.0)

20.0

In [4]:
h = 0.001
x = 2/3
(f(x+h)-f(x))/h

0.0029999999995311555

In [5]:
a = 2.0
b = -3.0
c = 10.0
d = a*b + c
print(d)

4.0


In [79]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None # Function that by default does nothin
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data + other.data, (self,other), '+')
        # Let's define what should happen to backward in addition:
        def _backward():
            self.grad += out.grad * 1.0 # Due to multivariable calculus gradients add
            other.grad += out.grad * 1.0
        out._backward = _backward
        return out

    def __radd__(self, other): # other + self
        return self + other
    
    def __mul__(self, other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data, (self,other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self,other): # other * self
        return self * other

    def __pow__(self, other): # self**other
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self, ), f'**{other}')
        
        def _backward():
            self.grad += other*self.data**(other-1) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __neg__(self): # -self
        return self * -1

    def __sub__(self,other): # self - other
        return self + (-other)
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out = Value(t, (self, ), 'tanh')
        def _backward():
            self.grad += (1-t**2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad # don't forget the chain rule!!
        out._backward = _backward
        return out

    def backward(self):
        # Build the topological graph:
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        # To begin the process ∂self/∂self = 1, so:
        self.grad = 1.0
        build_topo(self)
        for node in reversed(topo):
            node._backward() # Call the specific _backward() function of every operation


In [7]:
a=Value(2.0,label='a')
b=Value(4.0,label='b')

In [8]:
from graphviz import Digraph

def trace(root):
    # Helper function to build a set of all the nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # Left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op: # What? n._op is a string (I guess if it's non_empty it's true)
            dot.node(name = uid + n._op, label = n._op)
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

In [9]:
from IPython.display import display, HTML

def show_dot(root, width=1800):
    dot = draw_dot(root)

    svg = dot.pipe(format="svg").decode("utf-8")

    svg = svg.replace(
        "<svg ",
        f'<svg style="width:{width}px; height:auto; max-width:none;" ',
        1
    )

    display(HTML(f"""
    <div style="
        width: 100%;
        overflow-x: auto;
        overflow-y: hidden;
    ">
        {svg}
    </div>
    """))

In [10]:
# Time to model a neuron:
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1, w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neural-net
b = Value(6.8813735870195432, label='b')

x1w1 = x1*w1; x1w1.label='x1*w1'
x2w2 = x2*w2; x2w2.label='x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label='x1*w1 + x2*w2'
# x1w1 + x2w2 with intermediate pointers
n = x1w1x2w2 + b; n.label='n'

# Activation function (Sigmoid, tanh, )
o = n.tanh()
o.label='o'
o.backward()

In [11]:
show_dot(o,3000)

In [12]:
# Time to model a neuron:
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1, w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neural-net
b = Value(6.8813735870195432, label='b')

x1w1 = x1*w1; x1w1.label='x1*w1'
x2w2 = x2*w2; x2w2.label='x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label='x1*w1 + x2*w2'
# x1w1 + x2w2 with intermediate pointers
n = x1w1x2w2 + b; n.label='n'

# Activation function (Sigmoid, tanh, )
# Define o as a tanh now:

o = ((2*n).exp()-1)/((2*n).exp()+1)
o.label='o'
o.backward()

In [13]:
# Quick example with pyTorch:
import torch

x1 = torch.Tensor([2.0]).double(); x1.requires_grad = True
x2 = torch.Tensor([0.0]).double(); x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double(); w1.requires_grad = True
w2 = torch.Tensor([1.0]).double(); w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double(); b.requires_grad = True

n = x1*w1 + x2*w2 + b

o = torch.tanh(n)
o.backward()
print(o.item())
print('-----')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

0.7071066904050358
-----
x2 0.5000001283844369
w2 0.0
x1 -1.5000003851533106
w1 1.0000002567688737


In [110]:
class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        # w * x + b (dot product)
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
            

class MLP:
    def __init__(self, nin, nouts):
        size = [nin] + nouts # [3] + [4, 4, 1] = [3, 4, 4, 1]
        self.layers = [Layer(size[i], size[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [304]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4,4,1])
n(x)

Value(data=0.7150287890806603)

In [305]:
# Very simple classifier, define the data:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, 1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [340]:
# Actually make it respectable:

for k in range(20):

    # forward pass:
    ypred = [n(x) for x in xs]
    loss = sum((yout-ygt)**2 for ygt, yout in zip(ys,ypred))

    # backward pass:
    # DON'T forget to zero-grad! We want after each forward pass for the gradient to be 0, we don't know yet how they contribute
    # to the loss, we won't know until we compute loss.backward()!!
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    # update (gradient descent)
    for p in n.parameters():
        p.data += -0.15 * p.grad

    print(k, loss.data)

0 0.00024141757750671555
1 0.00024104623085550452
2 0.00024067600545789143
3 0.0002403068962781871
4 0.00023993889831068588
5 0.00023957200657946396
6 0.00023920621613815251
7 0.00023884152206969493
8 0.00023847791948616788
9 0.0002381154035285462
10 0.0002377539693664905
11 0.00023739361219813607
12 0.00023703432724990153
13 0.00023667610977625
14 0.00023631895505950455
15 0.00023596285840964446
16 0.00023560781516410897
17 0.00023525382068756475
18 0.00023490087037174735
19 0.00023454895963524508


In [343]:
show_dot(loss,4000)